# 07 Silver — Dimensões Conformadas

## Propósito

Este notebook define e materializa as **dimensões conformadas** (conformed dimensions) do modelo Silver a partir das tabelas Bronze disponíveis no dataset `daniellmattos/tse-analysis-eleicoes-2026`.

### Objetivos:
- **Inventário Bronze:** Listar todos os arquivos Parquet disponíveis e agrupá-los por tabela.
- **Matriz entidade-chave:** Mapear cada tabela Bronze → chave natural → grão → joins possíveis.
- **Dimensões conformadas:** Materializar `dim_election`, `dim_cargo`, `dim_partido` como Silver Parquet ZSTD.
- **Relatório de chaves órfãs:** Cruzar FK das tabelas fato vs PK das dimensões — apenas contagens, **sem descarte de dados**.
- **Manifesto de saída:** Sumário JSON com tabelas, contagens de linhas e timestamp de execução.

### Tabelas Bronze de entrada:
| Tabela | Fonte |
|---|---|
| `fact_urna_logs` | Logs da urna eletrônica (15 colunas) |
| `fact_boletim_urna` | Boletim de urna (45 colunas) |
| `fact_votacao_secao` | Apuração por seção (26 colunas) |
| `fact_detalhe_votacao_secao` | Detalhe votação seção (37 colunas) |
| `dim_candidato_munzona` | Candidatos por município/zona (50 colunas) |
| `dim_partido_munzona` | Partidos por município/zona (38 colunas) |
| `fact_eleitorado_perfil_secao` | Perfil do eleitorado por seção (30 colunas) |
| `dim_local_votacao_secao` | Locais de votação (41 colunas) |
| `fact_comparecimento_abstencao` | Comparecimento e abstenção (45 colunas) |
| `bridge_correspondencia_esperada` | Correspondências esperadas (21 colunas) |
| `bridge_correspondencia_efetivada` | Correspondências efetivadas (36 colunas) |

In [ ]:
# Cell 2 — Setup
import importlib.util
import json
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

for _pkg in ("polars",):
    if importlib.util.find_spec(_pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

import polars as pl

ELECTION_YEAR: int = 2026
SILVER_OUT = Path("/kaggle/working/silver")

# Locate BRONZE_BASE with robust search under /kaggle/input
BRONZE_BASE = Path("/kaggle/input/tse-analysis-eleicoes-2026")
candidate_bases = [
    Path("/kaggle/input/tse-analysis-eleicoes-2026"),
    Path("/kaggle/input/datasets/daniellmattos/tse-analysis-eleicoes-2026"),
]
for cand in candidate_bases:
    if cand.exists():
        BRONZE_BASE = cand
        break
else:
    # Locate by manifest checkpoint or parquet parent
    if Path("/kaggle/input").exists():
        for cp in Path("/kaggle/input").rglob("_checkpoints"):
            if cp.is_dir():
                BRONZE_BASE = cp.parent
                break
        else:
            for pq_file in Path("/kaggle/input").rglob("*.parquet"):
                # If path has e.g. .../tse-analysis-eleicoes-2026/table/part.parquet
                for parent in pq_file.parents:
                    if parent.name == "tse-analysis-eleicoes-2026":
                        BRONZE_BASE = parent
                        break
                if BRONZE_BASE.exists():
                    break

print(f"Polars version : {pl.__version__}")
print(f"ELECTION_YEAR  : {ELECTION_YEAR}")
print(f"Active BRONZE_BASE  : {BRONZE_BASE} (exists: {BRONZE_BASE.exists()})")
print(f"SILVER_OUT          : {SILVER_OUT}")

In [ ]:
# Cell 3 — Bronze Inventory
# List all Parquet files recursively and group by top-level table folder

from collections import defaultdict

all_parquets = sorted(BRONZE_BASE.rglob("*.parquet"))
print(f"Total Parquet files found under BRONZE_BASE: {len(all_parquets)}\n")

table_files: dict[str, list[Path]] = defaultdict(list)
for p in all_parquets:
    # Top-level subfolder relative to BRONZE_BASE is the table name
    rel = p.relative_to(BRONZE_BASE)
    table_name = rel.parts[0]
    table_files[table_name].append(p)

print(f"{'Table':<45} {'Files':>6} {'Sample path'}")
print("-" * 100)
for tname, files in sorted(table_files.items()):
    sample = str(files[0].relative_to(BRONZE_BASE))
    print(f"{tname:<45} {len(files):>6}   {sample}")

AVAILABLE_TABLES = set(table_files.keys())
print(f"\nAvailable tables: {sorted(AVAILABLE_TABLES)}")

In [ ]:
# Cell 4 — Load known Bronze dim/fact tables and inspect schema + shape

KNOWN_TABLES = [
    "dim_candidato_munzona",
    "dim_partido_munzona",
    "dim_local_votacao_secao",
    "fact_votacao_secao",
    "fact_detalhe_votacao_secao",
    "fact_detalhe_votacao_munzona",
    "fact_boletim_urna",
    "fact_comparecimento_abstencao",
    "fact_comparecimento_abstencao_pcd",
    "fact_comparecimento_abstencao_tte",
    "fact_eleitorado_perfil_secao",
    "fact_eleitorado_perfil_munzona",
    "fact_eleitorado_deficiencia",
    "fact_transferencia_temporaria_secao",
    "bridge_correspondencia_esperada",
    "bridge_correspondencia_efetivada",
    "fact_urna_logs",
]

loaded: dict[str, pl.DataFrame] = {}

for tname in KNOWN_TABLES:
    tpath = BRONZE_BASE / tname
    if not tpath.exists():
        print(f"[SKIP]  {tname:<45} — folder does not exist")
        continue
    try:
        pqs = sorted(tpath.rglob("*.parquet"))
        if not pqs:
            print(f"[SKIP]  {tname:<45} — no parquet files found")
            continue
        lf = pl.scan_parquet([str(p) for p in pqs])
        total_rows = lf.select(pl.len()).collect().item()
        # For large tables, sample 100k rows to keep memory usage low
        if total_rows > 100_000:
            df = lf.head(100_000).collect()
            print(f"[OK]    {tname:<45} total_rows={total_rows} (sampled 100k for schema inspection)")
        else:
            df = lf.collect()
            print(f"[OK]    {tname:<45} shape={df.shape}")
        loaded[tname] = df
        schema_preview = ", ".join(f"{c}: {t}" for c, t in list(df.schema.items())[:6])
        print(f"        schema (first 6): {schema_preview}")
        if df.shape[1] > 6:
            print(f"        ... +{df.shape[1]-6} more columns")
    except Exception as exc:
        print(f"[ERROR] {tname:<45} — {exc}")

print(f"\nSuccessfully inspected {len(loaded)}/{len(KNOWN_TABLES)} tables.")

In [ ]:
# Cell 5 — Entity-Key Matrix
# table | natural_key | grain | row_count | join_possible_to

MATRIX = [
    {
        "table": "dim_election",
        "natural_key": "election_year, turn",
        "grain": "election_year × turn",
        "join_possible_to": "ALL fact tables",
    },
    {
        "table": "dim_cargo",
        "natural_key": "CD_CARGO / CD_CARGO_PERGUNTA",
        "grain": "cargo (role/position)",
        "join_possible_to": "fact_votacao_secao, fact_boletim_urna, fact_detalhe_votacao_secao",
    },
    {
        "table": "dim_partido",
        "natural_key": "NR_PARTIDO (cd_partido)",
        "grain": "partido (political party)",
        "join_possible_to": "dim_candidato_munzona, fact_votacao_secao, fact_boletim_urna",
    },
    {
        "table": "dim_candidato_munzona",
        "natural_key": "SQ_CANDIDATO, CD_MUNICIPIO, NR_ZONA",
        "grain": "candidate × municipality × zone × turn",
        "join_possible_to": "fact_votacao_secao (SQ_CANDIDATO)",
    },
    {
        "table": "dim_partido_munzona",
        "natural_key": "NR_PARTIDO, CD_MUNICIPIO, NR_ZONA, NR_TURNO",
        "grain": "party × municipality × zone × turn",
        "join_possible_to": "fact_votacao_secao, fact_boletim_urna",
    },
    {
        "table": "dim_local_votacao_secao",
        "natural_key": "CD_MUNICIPIO, NR_ZONA, NR_SECAO",
        "grain": "section polling place",
        "join_possible_to": "fact_eleitorado_perfil_secao, fact_votacao_secao",
    },
    {
        "table": "fact_urna_logs",
        "natural_key": "cd_municipio, nr_zona, nr_secao, file_seq",
        "grain": "urna log line",
        "join_possible_to": "dim_election, dim_local_votacao_secao",
    },
    {
        "table": "fact_votacao_secao",
        "natural_key": "NR_TURNO, CD_MUNICIPIO, NR_ZONA, NR_SECAO, CD_CARGO, NR_VOTAVEL",
        "grain": "section × cargo × votable",
        "join_possible_to": "dim_election, dim_cargo, dim_partido, dim_candidato_munzona",
    },
    {
        "table": "fact_boletim_urna",
        "natural_key": "NR_TURNO, CD_MUNICIPIO, NR_ZONA, NR_SECAO, CD_CARGO_PERGUNTA, NR_VOTAVEL",
        "grain": "BU section × cargo × votable",
        "join_possible_to": "dim_election, dim_cargo, dim_partido",
    },
    {
        "table": "fact_comparecimento_abstencao",
        "natural_key": "NR_TURNO, SG_UF, CD_MUNICIPIO, NR_ZONA + demographic attrs",
        "grain": "zone × demographic profile × turn",
        "join_possible_to": "dim_election",
    },
    {
        "table": "bridge_correspondencia_esperada",
        "natural_key": "CD_PLEITO, CD_MUNICIPIO, NR_ZONA, NR_SECAO, NR_URNA_ESPERADA",
        "grain": "expected urna correspondence",
        "join_possible_to": "bridge_correspondencia_efetivada, fact_boletim_urna",
    },
]

# Enrich with actual row counts where available
for row in MATRIX:
    tname = row["table"]
    if tname in loaded:
        row["row_count"] = loaded[tname].shape[0]
    else:
        row["row_count"] = None

# Print formatted matrix
col_widths = {"table": 35, "natural_key": 50, "grain": 36, "row_count": 12, "join_possible_to": 55}
header = " | ".join(k.upper().ljust(v) for k, v in col_widths.items())
sep = "-+-".join("-" * v for v in col_widths.values())
print(header)
print(sep)
for row in MATRIX:
    rc = str(row.get("row_count", "n/a")) if row.get("row_count") is not None else "n/a"
    line = " | ".join([
        row["table"].ljust(col_widths["table"]),
        row["natural_key"].ljust(col_widths["natural_key"]),
        row["grain"].ljust(col_widths["grain"]),
        rc.rjust(col_widths["row_count"]),
        row["join_possible_to"].ljust(col_widths["join_possible_to"]),
    ])
    print(line)

In [ ]:
# Cell 6 — dim_election
# Synthetic dimension: one row per (election_year, turn)

dim_election_out = SILVER_OUT / "dim_election" / f"election_year={ELECTION_YEAR}"
dim_election_out.mkdir(parents=True, exist_ok=True)

dim_election = pl.DataFrame({
    "election_year": pl.Series([2026, 2026], dtype=pl.Int16),
    "turn": pl.Series([1, 2], dtype=pl.Int8),
    "election_label": pl.Series(
        ["Eleições Gerais 2026 — 1º Turno", "Eleições Gerais 2026 — 2º Turno"],
        dtype=pl.String,
    ),
    "election_date_turn1": pl.Series(["2026-10-04", "2026-10-04"], dtype=pl.String),
    "election_date_turn2": pl.Series(["2026-10-25", "2026-10-25"], dtype=pl.String),
    "cd_pleito_turn1": pl.Series([3220, 3220], dtype=pl.Int32),
})

out_path = dim_election_out / "part-00000.parquet"
dim_election.write_parquet(
    str(out_path),
    compression="zstd",
    compression_level=5,
    use_pyarrow=False,
)

print(f"dim_election written: {out_path}")
print(f"Shape: {dim_election.shape}")
print(dim_election)

In [ ]:
# Cell 7 — dim_cargo
# Derive from fact_votacao_secao (CD_CARGO, DS_CARGO) or fact_boletim_urna (CD_CARGO_PERGUNTA, DS_CARGO_PERGUNTA)
# Fall back to fact_detalhe_votacao_secao or canonical list if needed

dim_cargo_out = SILVER_OUT / "dim_cargo"
dim_cargo_out.mkdir(parents=True, exist_ok=True)

cargo_df: pl.DataFrame | None = None

for src_table, cargo_col, desc_col in [
    ("fact_votacao_secao", "CD_CARGO", "DS_CARGO"),
    ("fact_detalhe_votacao_secao", "CD_CARGO", "DS_CARGO"),
    ("fact_boletim_urna", "CD_CARGO_PERGUNTA", "DS_CARGO_PERGUNTA"),
]:
    tpath = BRONZE_BASE / src_table
    pqs = sorted(tpath.rglob("*.parquet")) if tpath.exists() else []
    if pqs:
        try:
            cargo_df = (
                pl.scan_parquet([str(p) for p in pqs])
                .select([cargo_col, desc_col])
                .collect()
                .rename({cargo_col: "cd_cargo", desc_col: "ds_cargo"})
                .unique(subset=["cd_cargo"])
                .sort("cd_cargo")
            )
            if cargo_df["cd_cargo"].dtype != pl.Int32:
                cargo_df = cargo_df.with_columns(pl.col("cd_cargo").cast(pl.Int32, strict=False))
            print(f"dim_cargo sourced from: {src_table} ({cargo_col}/{desc_col}) — {cargo_df.shape[0]} unique cargos")
            break
        except Exception as exc:
            print(f"Could not extract cargo from {src_table}: {exc}")

if cargo_df is None:
    # Fallback: build a known cargo lookup from TSE domain knowledge
    print("WARNING: No source table found for dim_cargo — using TSE canonical cargo list")
    cargo_df = pl.DataFrame({
        "cd_cargo": pl.Series(
            [1, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13],
            dtype=pl.Int32,
        ),
        "ds_cargo": pl.Series([
            "PRESIDENTE",
            "GOVERNADOR",
            "SENADOR",
            "DEPUTADO FEDERAL",
            "DEPUTADO ESTADUAL",
            "DEPUTADO DISTRITAL",
            "1º SUPLENTE SENADOR",
            "2º SUPLENTE SENADOR",
            "VICE-GOVERNADOR",
            "VICE-PRESIDENTE",
            "DEPUTADO FEDERAL (VAGA REMANESCENTE)",
        ], dtype=pl.String),
    })

out_path = dim_cargo_out / "part-00000.parquet"
cargo_df.write_parquet(
    str(out_path),
    compression="zstd",
    compression_level=5,
    use_pyarrow=False,
)

print(f"dim_cargo written: {out_path}")
print(f"Shape: {cargo_df.shape}")
print(cargo_df)

In [ ]:
# Cell 8 — dim_partido
# Deduplicate by NR_PARTIDO (cd_partido) from dim_partido_munzona

dim_partido_out = SILVER_OUT / "dim_partido"
dim_partido_out.mkdir(parents=True, exist_ok=True)

partido_df: pl.DataFrame | None = None

# Column mapping candidates from dim_partido_munzona schema (38 cols)
PARTIDO_CANDIDATES = [
    # (source_table, partido_num_col, partido_sg_col, partido_nm_col)
    ("dim_partido_munzona", "NR_PARTIDO", "SG_PARTIDO", "NM_PARTIDO"),
    ("dim_candidato_munzona", "NR_PARTIDO", "SG_PARTIDO", "NM_PARTIDO"),
    ("fact_votacao_secao", "NR_VOTAVEL", "SG_PARTIDO", "NM_VOTAVEL"),  # partial
    ("fact_boletim_urna", "NR_PARTIDO", "SG_PARTIDO", "NM_PARTIDO"),
]

for src_table, nr_col, sg_col, nm_col in PARTIDO_CANDIDATES:
    tpath = BRONZE_BASE / src_table
    pqs = sorted(tpath.rglob("*.parquet")) if tpath.exists() else []
    if pqs:
        try:
            raw = (
                pl.scan_parquet([str(p) for p in pqs])
                .select([nr_col, sg_col, nm_col])
                .collect()
                .rename({
                    nr_col: "cd_partido",
                    sg_col: "sg_partido",
                    nm_col: "nm_partido",
                })
            )
            if raw["cd_partido"].dtype != pl.Int32:
                raw = raw.with_columns(pl.col("cd_partido").cast(pl.Int32, strict=False))
            raw = (
                raw
                .filter(pl.col("cd_partido").is_not_null() & (pl.col("cd_partido") > 0))
                .unique(subset=["cd_partido"])
                .sort("cd_partido")
            )
            if raw.shape[0] > 0:
                partido_df = raw
                print(f"dim_partido sourced from: {src_table} ({nr_col}/{sg_col}/{nm_col}) — {raw.shape[0]} unique parties")
                break
        except Exception as exc:
            print(f"Could not extract partido from {src_table}: {exc}")

if partido_df is None:
    print("WARNING: No source table found for dim_partido — using canonical TSE 2026 party registry")
    partido_df = pl.DataFrame({
        "cd_partido": pl.Series([
            10, 11, 12, 13, 15, 16, 18, 19, 21, 22, 23, 25, 27, 28, 29, 30,
            33, 35, 36, 40, 43, 44, 45, 50, 55, 65, 70, 77, 80,
        ], dtype=pl.Int32),
        "sg_partido": pl.Series([
            "REPUBLICANOS", "PP", "PDT", "PT", "MDB", "PSTU", "REDE", "PODE",
            "PCB", "PL", "CIDADANIA", "PRD", "DC", "PRTB", "PCO", "NOVO",
            "MOBILIZA", "PMB", "AGIR", "PSB", "PV", "UNIÃO", "PSDB", "PSOL",
            "PSD", "PC do B", "AVANTE", "SOLIDARIEDADE", "UP",
        ], dtype=pl.String),
        "nm_partido": pl.Series([
            "REPUBLICANOS", "PROGRESSISTAS", "PARTIDO DEMOCRÁTICO TRABALHISTA",
            "PARTIDO DOS TRABALHADORES", "MOVIMENTO DEMOCRÁTICO BRASILEIRO",
            "PARTIDO SOCIALISTA DOS TRABALHADORES UNIFICADO", "REDE SUSTENTABILIDADE",
            "PODEMOS", "PARTIDO COMUNISTA BRASILEIRO", "PARTIDO LIBERAL",
            "CIDADANIA", "PARTIDO RENOVAÇÃO DEMOCRÁTICA", "DEMOCRACIA CRISTÃ",
            "PARTIDO RENOVADOR TRABALHISTA BRASILEIRO", "PARTIDO DA CAUSA OPERÁRIA",
            "PARTIDO NOVO", "MOBILIZAÇÃO NACIONAL", "PARTIDO DA MULHER BRASILEIRA",
            "AGIR", "PARTIDO SOCIALISTA BRASILEIRO", "PARTIDO VERDE",
            "UNIÃO BRASIL", "PARTIDO DA SOCIAL DEMOCRACIA BRASILEIRA",
            "PARTIDO SOCIALISMO E LIBERDADE", "PARTIDO SOCIAL DEMOCRÁTICO",
            "PARTIDO COMUNISTA DO BRASIL", "AVANTE", "SOLIDARIEDADE",
            "UNIDADE POPULAR",
        ], dtype=pl.String),
    })

out_path = dim_partido_out / "part-00000.parquet"
partido_df.write_parquet(
    str(out_path),
    compression="zstd",
    compression_level=5,
    use_pyarrow=False,
)

print(f"dim_partido written: {out_path}")
print(f"Shape: {partido_df.shape}")
print(partido_df.head(15))

In [ ]:
# Cell 9 — Orphan Key Report
# Cross-reference FK columns in fact tables vs PK of Silver dims
# IMPORTANT: counts only — no data is ever dropped

print("=" * 70)
print("ORPHAN KEY REPORT — FK vs Silver Dim PKs")
print("=" * 70)
print("NOTE: orphan counts are diagnostic only — no rows are dropped.\n")

# Reload dim Silver Parquets for FK checks
silver_dims: dict[str, pl.DataFrame] = {}
for dim_name in ("dim_election", "dim_cargo", "dim_partido"):
    dim_dir = SILVER_OUT / dim_name
    if dim_dir.exists():
        try:
            silver_dims[dim_name] = pl.read_parquet(str(dim_dir / "**" / "*.parquet"))
            print(f"[OK] Silver {dim_name}: {silver_dims[dim_name].shape[0]} rows")
        except Exception as exc:
            print(f"[ERR] Silver {dim_name}: {exc}")

print()

# --- dim_election FK check ---
dim_elec = silver_dims.get("dim_election")
if dim_elec is not None:
    valid_turns = set(dim_elec["turn"].to_list())
    for tname, turn_col in [
        ("fact_votacao_secao", "NR_TURNO"),
        ("fact_boletim_urna", "NR_TURNO"),
        ("fact_comparecimento_abstencao", "NR_TURNO"),
    ]:
        if tname in loaded and turn_col in loaded[tname].columns:
            df_t = loaded[tname]
            total = df_t.shape[0]
            orphans = df_t.filter(~pl.col(turn_col).cast(pl.Int8).is_in(list(valid_turns))).shape[0]
            pct = orphans / total * 100 if total > 0 else 0.0
            print(f"dim_election FK | {tname}.{turn_col}: {orphans}/{total} orphans ({pct:.2f}%)")

print()

# --- dim_cargo FK check ---
dim_cargo_silver = silver_dims.get("dim_cargo")
if dim_cargo_silver is not None:
    valid_cargos = set(dim_cargo_silver["cd_cargo"].to_list())
    for tname, cargo_col in [
        ("fact_votacao_secao", "CD_CARGO"),
        ("fact_detalhe_votacao_secao", "CD_CARGO"),
        ("fact_boletim_urna", "CD_CARGO_PERGUNTA"),
    ]:
        if tname in loaded and cargo_col in loaded[tname].columns:
            df_t = loaded[tname]
            total = df_t.shape[0]
            orphans = df_t.filter(~pl.col(cargo_col).cast(pl.Int32, strict=False).is_in(list(valid_cargos))).shape[0]
            pct = orphans / total * 100 if total > 0 else 0.0
            print(f"dim_cargo FK    | {tname}.{cargo_col}: {orphans}/{total} orphans ({pct:.2f}%)")

print()

# --- dim_partido FK check ---
dim_part_silver = silver_dims.get("dim_partido")
if dim_part_silver is not None and dim_part_silver.shape[0] > 0:
    valid_partidos = set(dim_part_silver["cd_partido"].to_list())
    for tname, partido_col in [
        ("dim_candidato_munzona", "NR_PARTIDO"),
        ("fact_boletim_urna", "NR_PARTIDO"),
        ("dim_partido_munzona", "NR_PARTIDO"),
    ]:
        if tname in loaded and partido_col in loaded[tname].columns:
            df_t = loaded[tname]
            total = df_t.shape[0]
            # Positive partido codes only (0 = brancos/nulos)
            df_pos = df_t.filter(pl.col(partido_col).cast(pl.Int32, strict=False) > 0)
            total_pos = df_pos.shape[0]
            orphans = df_pos.filter(
                ~pl.col(partido_col).cast(pl.Int32, strict=False).is_in(list(valid_partidos))
            ).shape[0]
            pct = orphans / total_pos * 100 if total_pos > 0 else 0.0
            print(
                f"dim_partido FK  | {tname}.{partido_col}: "
                f"{orphans}/{total_pos} orphans (of {total} total) ({pct:.2f}%)"
            )

print("\nOrphan report complete.")

In [ ]:
# Cell 10 — Manifest
# JSON summary: tables written, row counts, timestamp

manifest_tables: dict[str, int] = {}

# Enumerate Silver output parquets
for silver_pq in sorted(SILVER_OUT.rglob("*.parquet")):
    rel = silver_pq.relative_to(SILVER_OUT)
    table_name = rel.parts[0]
    try:
        df_check = pl.read_parquet(str(silver_pq))
        manifest_tables[table_name] = manifest_tables.get(table_name, 0) + df_check.shape[0]
    except Exception:
        manifest_tables[table_name] = -1

manifest = {
    "notebook": "07_silver_dimensions",
    "election_year": ELECTION_YEAR,
    "silver_base": str(SILVER_OUT),
    "bronze_tables_loaded": len(loaded),
    "silver_tables_written": manifest_tables,
    "total_silver_rows": sum(v for v in manifest_tables.values() if v >= 0),
    "generated_at": datetime.now(UTC).isoformat(),
    "polars_version": pl.__version__,
}

print(json.dumps(manifest, indent=2, ensure_ascii=False))

# Also persist to /kaggle/working
manifest_path = Path("/kaggle/working/07_silver_dimensions_manifest.json")
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nManifest written to: {manifest_path}")

## Limitações e Restrições

### Cobertura de Dados
- As dimensões Silver (`dim_cargo`, `dim_partido`) são derivadas das tabelas Bronze disponíveis no dataset. Se uma tabela Bronze não estiver presente (p. ex. porque o pipeline de ingestão ainda não a processou), a dimensão pode ser construída a partir de uma fallback canônica do TSE ou ficar vazia.
- `dim_election` é uma dimensão **sintética** (não derivada de nenhum CSV real do TSE); reflete o conhecimento a priori do calendário eleitoral 2026 (1º turno: 04/out, 2º turno: 25/out).

### Particionamento
- As dimensões Silver **não são particionadas** por `election_year/turn/uf` (seria ineficiente para tabelas pequenas como `dim_cargo` e `dim_partido`). Apenas `dim_election` usa a partição `election_year=2026/` por convenção de path.
- As tabelas fato Silver (p. ex. `fact_urna_event`) seguem a política padrão de particionamento `(election_year, turn, uf)`.

### Chaves Naturais vs Surrogate Keys
- Este notebook trabalha **exclusivamente com chaves naturais** do TSE (CD_CARGO, NR_PARTIDO, etc.). Não são geradas surrogate keys (IDs inteiros sequenciais), pois as chaves naturais TSE são estáveis entre turnos/anos.

### Chaves Órfãs
- O relatório de chaves órfãs (Cell 9) é **diagnóstico apenas** — nenhum dado é descartado.
- Registros com `NR_PARTIDO = 0` correspondem a votos em branco/nulos e são explicitamente excluídos das verificações de FK (são válidos por definição).

### Escopo
- Este notebook cobre **exclusivamente** as dimensões conformadas. As tabelas fato Silver (`fact_urna_event`, etc.) são materializadas em notebooks separados (05, 06).
- `dim_local_votacao_secao` e `dim_candidato_munzona` são preservadas em Bronze sem transformação neste notebook, pois são grandes demais para deduplicação simples sem uma chave composta clara.

### Reprodutibilidade
- 100% autocontido: nenhum `import tse_analysis`, nenhuma dependência de arquivos fora de `/kaggle/input/` e `/kaggle/working/`.
- Idempotente: pode ser re-executado sem efeitos colaterais (sobrescreve os mesmos arquivos Silver).